In [124]:
import scanpy as sc
import numpy as np 
import matplotlib.pyplot as plt
import multivelo as mv
import pandas as pd
import scvelo as scv
import anndata as ad
from pathlib import Path
from scipy.spatial.distance import cdist, euclidean
from scipy.optimize import linear_sum_assignment
import seaborn as sns
from scipy.stats import pearsonr, spearmanr
from matplotlib.collections import LineCollection
from sklearn.neighbors import NearestNeighbors
import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests
from matplotlib.lines import Line2D
import pickle as pkl
from cytotrace2_py.cytotrace2_py import *
from sklearn.linear_model import LogisticRegression

python_data_dir = r"C:\Users\Nikla\Documents\MoBi_Master_FS.3\Internship AG Höfer\DynaVelo_testing\data"

scv.settings.verbosity = 3
scv.settings.presenter_view = True
scv.set_figure_params('scvelo')
scv.settings.figdir = rf"{python_data_dir}\multivelo_comparison"
scv.settings.plot_prefix = ""
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 200)
np.set_printoptions(suppress=True)

## Load ATAC data and subset Peaks

In [40]:
adata_atac_full = sc.read_h5ad(fr"{python_data_dir}\all_atac_peaks_corr.h5ad")

In [49]:
peak_type_annotation = pd.read_csv(rf"{python_data_dir}\Exp4_ctr_peak_types.csv", header=0, index_col=0)

In [50]:
peak_type_annotation = peak_type_annotation[peak_type_annotation["peak_id"].isin(adata_atac_full.var_names.values)]
print("All indices present:", (peak_type_annotation["peak_id"].isin(adata_atac_full.var_names.values)).all())

All indices present: True


In [51]:
keep_peaks = peak_type_annotation.loc[peak_type_annotation["peakType"]=="Promoter"]  # left with roughly 27k peaks for promoters

In [52]:
keep_peaks 

,chromosome,start,stop,peakType,nearestTSS,nearestGene,peak_id
14,chr1,4497330,4497830,Promoter,uc007aez.2,Sox17,chr1:4497330-4497830
28,chr1,4785493,4785993,Promoter,uc007afd.4,Mrpl15,chr1:4785493-4785993
29,chr1,4806768,4807268,Promoter,uc007afg.1,Lypla1,chr1:4806768-4807268
30,chr1,4807561,4808061,Promoter,uc007afg.1,Lypla1,chr1:4807561-4808061
31,chr1,4855555,4856055,Promoter,uc007afi.2,Tcea1,chr1:4855555-4856055
...,...,...,...,...,...,...,...
210844,chrX,168674049,168674549,Promoter,uc009uxh.2,Msl3,chrX:168674049-168674549
210847,chrX,168794788,168795288,Promoter,uc009uxj.2,Arhgap6,chrX:168794788-168795288
210875,chrX,169319025,169319525,Promoter,uc009uxx.3,Hccs,chrX:169319025-169319525
210876,chrX,169319564,169320064,Promoter,uc009uxx.3,Hccs,chrX:169319564-169320064


In [54]:
adata_atac_full = adata_atac_full[:, adata_atac_full.var_names.isin(keep_peaks["peak_id"])]

## Load RNA data and subset to select HVGs

In [67]:
adata_rna = sc.read_h5ad(rf"{python_data_dir}\all_rna_counts.h5ad")

nn_idx_path = rf"{python_data_dir}\nn_idx.txt"
nn_dist_path = rf"{python_data_dir}\nn_dist.txt"
nn_cells_path = rf"{python_data_dir}\nn_cells.txt"

nn_idx = np.loadtxt(str(nn_idx_path), delimiter=',')
nn_dist = np.loadtxt(str(nn_dist_path), delimiter=',')
nn_cells = pd.Index(pd.read_csv(str(nn_cells_path), header=None)[0])
nn_cells = nn_cells.str.replace('-1', '', regex=False)

In [68]:
shared_cells = pd.Index(np.intersect1d(adata_rna.obs_names, adata_atac_full.obs_names))
#shared_genes = pd.Index(np.intersect1d(adata_rna.var_names, adata_atac_full.var_names))
adata_rna.obs.rename(columns={"cell_type": "celltype"}, inplace=True)

In [69]:
adata_rna = adata_rna[shared_cells,:]
adata_atac_full = adata_atac_full[shared_cells,:]

In [70]:
embedding_genes = pd.read_csv(rf"C:\Users\Nikla\Documents\MoBi_Master_FS.3\Internship AG Höfer\DynaVelo_preprocessing\data\Exp4_ctr_rna_pca_genes.csv", header=[0]).T.values.tolist()[0]
adata_rna.var['highly_variable'] = adata_rna.var_names.isin(embedding_genes)

In [ ]:
adata_rna = adata_rna[:,adata_rna.var['highly_variable']]  # Subset to the embedding genes only, use this to subset peaks

## Subset both RNA and ATAC to have common peaks/genes

In [76]:
embedding_peaks = keep_peaks[keep_peaks["nearestGene"].isin(adata_rna.var_names.values)]

In [78]:
embedding_peaks

,chromosome,start,stop,peakType,nearestTSS,nearestGene,peak_id
370,chr1,14310356,14310856,Promoter,uc007aix.1,Eya1,chr1:14310356-14310856
412,chr1,16104175,16104675,Promoter,uc007ajl.4,Rpl7,chr1:16104175-16104675
413,chr1,16104755,16105255,Promoter,uc007ajl.4,Rpl7,chr1:16104755-16105255
434,chr1,16619106,16619606,Promoter,uc007ajs.2,Ube2w,chr1:16619106-16619606
435,chr1,16621072,16621572,Promoter,uc007ajs.2,Ube2w,chr1:16621072-16621572
...,...,...,...,...,...,...,...
210807,chrX,167208955,167209455,Promoter,uc009uwx.1,Tmsb4x,chrX:167208955-167209455
210808,chrX,167209584,167210084,Promoter,uc009uwx.1,Tmsb4x,chrX:167209584-167210084
210809,chrX,167210141,167210641,Promoter,uc009uwx.1,Tmsb4x,chrX:167210141-167210641
210810,chrX,167210655,167211155,Promoter,uc009uwx.1,Tmsb4x,chrX:167210655-167211155


In [80]:
adata_atac_sub = adata_atac_full[:,embedding_peaks["peak_id"]]

In [82]:
atac_new_X = np.zeros(shape=adata_rna.shape)

In [ ]:
(adata_rna.obs_names.values == adata_atac_sub.obs_names.values).all()  # indices are in the same order!

True

In [122]:
import scipy.sparse as sp

df = embedding_peaks[["peak_id", "nearestGene"]].drop_duplicates()

peak_idx = pd.Index(adata_atac_sub.var_names).get_indexer(df["peak_id"])
gene_idx = pd.Index(adata_rna.var_names).get_indexer(df["nearestGene"])

keep = (peak_idx >= 0) & (gene_idx >= 0)

M = sp.csr_matrix(
    (np.ones(keep.sum(), dtype=np.float32), (peak_idx[keep], gene_idx[keep])),
    shape=(adata_atac_sub.n_vars, adata_rna.n_vars),
)

atac_new_X = adata_atac_sub.X @ M
atac_new_X = atac_new_X.toarray()

In [125]:
adata_atac_pmtrs = ad.AnnData(X=atac_new_X)
adata_atac_pmtrs.var_names = adata_rna.var_names
adata_atac_pmtrs.obs_names = adata_rna.obs_names

In [128]:
adata_atac_pmtrs.write_h5ad(rf"{python_data_dir}\pmtr_peak_sub\atac.h5ad")
adata_rna.write_h5ad(rf"{python_data_dir}\pmtr_peak_sub\rna.h5ad")

## Work with aligned data subsets